<a href="https://colab.research.google.com/github/Pranavv-07/24B11CS355_DriveReady_OPC_2026/blob/main/Day_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-groq

In [2]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")   # LangChain reads it from here

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.3)

In [5]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

SYSTEM = """ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80 each.
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time or place. If it isn't in FACTS,
        say "I'll check with the owner." Work out totals carefully.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),          # the chat so far goes here
    ("human", "{question}"),
])
chain = prompt | llm | StrOutputParser()

history = []
def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Pranav. Do you make Vanilla cup cakes?"))
print(ask("What's my name?"))

I'll check with the owner.
Your name is Pranav.


In [6]:
history

[('human', "Hi, I'm Pranav. Do you make Vanilla cup cakes?"),
 ('ai', "I'll check with the owner."),
 ('human', "What's my name?"),
 ('ai', 'Your name is Pranav.')]

In [7]:
import csv

def run_tests(path: str = "/content/test.csv") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0
    for row in rows:
        answer = chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])
    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")



In [9]:
# Create a dummy test.csv file so run_tests has data to run against
import csv

test_data = [
    {"question": "Do you make eggless cakes?", "must_include": "yes"},
    {"question": "How much is a Chocolate cake?", "must_include": "650"},
    {"question": "Can I get a delivery today?", "must_include": "owner"}
]

with open("/content/test.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["question", "must_include"])
    writer.writeheader()
    writer.writerows(test_data)

# Now run the test suite
run_tests()

PASS | Do you make eggless cakes? → Yes! All our cakes – plum, chocolate and cupcakes – are available in e
PASS | How much is a Chocolate cake? → A Chocolate cake costs Rs 650. Let me know if you’d like to add anythi
PASS | Can I get a delivery today? → I’ll check with the owner and get back to you shortly!

3/3 passed (100%)


In [10]:
from langchain.agents import create_agent

agent_order = create_agent(model=llm, tools=[], system_prompt=SYSTEM)   # tools come tomorrow
result_order = agent.invoke({"messages": [{"role": "user", "content": "Do you make eggless cakes?"}]})
print(result["messages"][-1].content)

NameError: name 'agent' is not defined